# 06 — Civil Engineering Copilot, end to end

> **EDUCATIONAL TOY — NOT PRODUCTION**

This notebook builds a tiny, transparent version of the whole idea. It uses no network, credentials, or external services. The real application does not import this notebook.

## The whole flow

![End-to-end Civil Copilot architecture](../docs/images/civil-copilot-architecture-overview.png)

*The overview separates preparation from question-time retrieval, tools, and agents.*

Prepare a small connected project **before** questions are asked:

source records → validate → chunk → exact/sparse/dense indexes → relationship graph

Then answer questions:

question → retrieve evidence → choose Fast RAG, Graph RAG, or bounded Agentic RAG → cite or abstain

## What you will see

We will use two connected examples:

1. **RFI-087 decision and downstream impact** — Fast RAG and Graph RAG.
2. **ACT-STEEL-009 delay investigation** — typed tools, Document Specialist, Schedule Specialist, Risk Specialist, bounded ReAct, memory, citations, and evaluation.

## Small glossary

- **Chunk:** a searchable passage that still knows its parent record and source.
- **Exact retrieval:** finds identifiers such as RFI-087.
- **Sparse retrieval:** scores shared words; it is useful for precise terms.
- **Dense retrieval:** compares small meaning vectors.
- **Rerank:** reorders fused candidates using transparent project rules.
- **RAG:** retrieve evidence before answering.
- **Graph RAG:** follow stored relationships rather than guessing connections.
- **Tool:** one controlled, typed operation.
- **Memory:** approved presentation preferences, never project truth.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import re
from collections import Counter, defaultdict, deque
from collections.abc import Sequence
from html import escape
from typing import Any, Literal

from IPython.display import HTML, display
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import BaseTool, tool
from pydantic import BaseModel, Field

TOKEN = re.compile(r"[a-z0-9-]+")
IDENTIFIER = re.compile(r"\b[A-Z]{2,}(?:-[A-Z0-9]+)+\b")
PROJECT_ID = "TOY-BLR-STEEL"
PROJECT_SCOPE = "project:toy-blr-steel"

In [2]:
records = [
    {
        "record_id": "PUBLIC-IS-800-PREVIEW",
        "record_type": "public_reference",
        "title": "IS 800 public catalogue preview",
        "text": "Official public preview and catalogue material only; this is not the complete Indian Standard.",
        "status": "preview",
        "revision": "catalogue",
        "origin": "public_official",
        "scopes": ["public"],
        "metadata": {},
    },
    {
        "record_id": "SPEC-STEEL-001",
        "record_type": "specification",
        "title": "Structural steel project specification",
        "text": "The synthetic project specification references the project code register and connection requirements.",
        "status": "current",
        "revision": "1",
        "origin": "synthetic_academic_demo",
        "scopes": [PROJECT_SCOPE],
        "metadata": {},
    },
    {
        "record_id": "DRAW-S-204-R3",
        "record_type": "drawing",
        "title": "S-204 framing plan revision 3",
        "text": "Superseded framing plan for grid 4 showing connection C17 before plate PL-17B was approved.",
        "status": "superseded",
        "revision": "3",
        "origin": "synthetic_academic_demo",
        "scopes": [PROJECT_SCOPE],
        "metadata": {"document_number": "S-204"},
    },
    {
        "record_id": "DRAW-S-204-R5",
        "record_type": "drawing",
        "title": "S-204 framing plan revision 5",
        "text": "Current framing plan for grid 4 includes approved plate PL-17B at connection C17.",
        "status": "current",
        "revision": "5",
        "origin": "synthetic_academic_demo",
        "scopes": [PROJECT_SCOPE],
        "metadata": {"document_number": "S-204"},
    },
    {
        "record_id": "RFI-087",
        "record_type": "rfi",
        "title": "Connection C17 clarification",
        "text": "RFI-087 approved plate PL-17B for connection C17. The decision was incorporated in S-204 revision 5.",
        "status": "closed",
        "revision": "response-1",
        "origin": "synthetic_academic_demo",
        "scopes": [PROJECT_SCOPE],
        "metadata": {},
    },
    {
        "record_id": "ACT-STEEL-009",
        "record_type": "schedule_activity",
        "title": "Level 2 zone 3 steel erection",
        "text": "ACT-STEEL-009 was blocked until the revised S-204 drawing was issued. Planned duration is seven days.",
        "status": "in_progress",
        "revision": "baseline-2",
        "origin": "synthetic_academic_demo",
        "scopes": [PROJECT_SCOPE],
        "metadata": {
            "critical": True,
            "duration_days": 7,
            "blocked_by_ids": ["RFI-087"],
            "drawing_numbers": ["S-204"],
        },
    },
    {
        "record_id": "RISK-DELAY-001",
        "record_type": "risk",
        "title": "Critical steel erection delay risk",
        "text": "The delay risk is high while the critical steel activity remains blocked by unresolved design information.",
        "status": "mitigated",
        "revision": "2",
        "origin": "synthetic_academic_demo",
        "scopes": [PROJECT_SCOPE],
        "metadata": {"severity": "high"},
    },
]

relationships = [
    ("RFI-087", "REFERENCES", "DRAW-S-204-R3", "RFI response"),
    ("RFI-087", "CHANGES_OR_CLARIFIES", "DRAW-S-204-R5", "RFI response"),
    ("RFI-087", "AFFECTS", "ACT-STEEL-009", "RFI response"),
    ("DRAW-S-204-R5", "REVISES", "DRAW-S-204-R3", "drawing register"),
    ("ACT-STEEL-009", "GOVERNED_BY", "SPEC-STEEL-001", "schedule mapping"),
    ("RISK-DELAY-001", "DERIVED_FROM", "RFI-087", "risk register"),
    ("RISK-DELAY-001", "AFFECTS", "ACT-STEEL-009", "risk register"),
]

In [3]:
origin_summary = Counter(record["origin"] for record in records)
[
    {
        "record_id": record["record_id"],
        "origin": record["origin"],
        "source_boundary": (
            "official preview only"
            if record["origin"] == "public_official"
            else "synthetic academic project"
        ),
    }
    for record in records
], dict(origin_summary)

([{'record_id': 'PUBLIC-IS-800-PREVIEW',
   'origin': 'public_official',
   'source_boundary': 'official preview only'},
  {'record_id': 'SPEC-STEEL-001',
   'origin': 'synthetic_academic_demo',
   'source_boundary': 'synthetic academic project'},
  {'record_id': 'DRAW-S-204-R3',
   'origin': 'synthetic_academic_demo',
   'source_boundary': 'synthetic academic project'},
  {'record_id': 'DRAW-S-204-R5',
   'origin': 'synthetic_academic_demo',
   'source_boundary': 'synthetic academic project'},
  {'record_id': 'RFI-087',
   'origin': 'synthetic_academic_demo',
   'source_boundary': 'synthetic academic project'},
  {'record_id': 'ACT-STEEL-009',
   'origin': 'synthetic_academic_demo',
   'source_boundary': 'synthetic academic project'},
  {'record_id': 'RISK-DELAY-001',
   'origin': 'synthetic_academic_demo',
   'source_boundary': 'synthetic academic project'}],
 {'public_official': 1, 'synthetic_academic_demo': 6})

In [4]:
record_ids = {record["record_id"] for record in records}
assert len(record_ids) == len(records)
assert all(source in record_ids and target in record_ids for source, _, target, _ in relationships)
assert all(record["origin"] in {"public_official", "synthetic_academic_demo"} for record in records)
assert all(record["scopes"] for record in records)
validation = {
    "records": len(records),
    "relationships": len(relationships),
    "dangling_relationships": 0,
    "origin_labels_complete": True,
}
validation

{'records': 7,
 'relationships': 7,
 'dangling_relationships': 0,
 'origin_labels_complete': True}

# 1. Offline/index-time preparation

![Data ingestion architecture](../docs/images/data-ingestion-architecture.png)

*Source records are checked, split into searchable passages, and published with their origin labels.*

“Offline” here means work done before a user asks a question. We create citable chunks and purpose-specific indexes once, then reuse them.

In [5]:
chunks = [
    {
        "chunk_id": f'{record["record_id"]}-chunk-1',
        "record_id": record["record_id"],
        "text": f'{record["title"]}. {record["text"]}',
        "origin": record["origin"],
        "status": record["status"],
        "scopes": record["scopes"],
        "metadata": record["metadata"],
    }
    for record in records
]
assert {chunk["record_id"] for chunk in chunks} <= record_ids
chunks[:2]

[{'chunk_id': 'PUBLIC-IS-800-PREVIEW-chunk-1',
  'record_id': 'PUBLIC-IS-800-PREVIEW',
  'text': 'IS 800 public catalogue preview. Official public preview and catalogue material only; this is not the complete Indian Standard.',
  'origin': 'public_official',
  'status': 'preview',
  'scopes': ['public'],
  'metadata': {}},
 {'chunk_id': 'SPEC-STEEL-001-chunk-1',
  'record_id': 'SPEC-STEEL-001',
  'text': 'Structural steel project specification. The synthetic project specification references the project code register and connection requirements.',
  'origin': 'synthetic_academic_demo',
  'status': 'current',
  'scopes': ['project:toy-blr-steel'],
  'metadata': {}}]

In [6]:
tokenized_chunks = {
    chunk["chunk_id"]: TOKEN.findall(chunk["text"].lower())
    for chunk in chunks
}
inverted_index: dict[str, set[str]] = defaultdict(set)
for chunk_id, tokens in tokenized_chunks.items():
    for token in set(tokens):
        inverted_index[token].add(chunk_id)

document_frequency = {token: len(ids) for token, ids in inverted_index.items()}
average_length = sum(map(len, tokenized_chunks.values())) / len(tokenized_chunks)
{"sparse_terms": len(inverted_index), "average_chunk_tokens": round(average_length, 1)}

{'sparse_terms': 75, 'average_chunk_tokens': 18.9}

In [7]:
DENSE_DIMENSION = 32

def dense_vector(text: str) -> list[float]:
    vector = [0.0] * DENSE_DIMENSION
    for token in TOKEN.findall(text.lower()):
        digest = hashlib.sha256(token.encode()).digest()
        index = int.from_bytes(digest[:4], "big") % DENSE_DIMENSION
        vector[index] += 1.0 if digest[4] % 2 else -1.0
    norm = math.sqrt(sum(value * value for value in vector)) or 1.0
    return [value / norm for value in vector]

dense_index = {
    chunk["chunk_id"]: dense_vector(chunk["text"])
    for chunk in chunks
}
{"dense_vectors": len(dense_index), "dimension": DENSE_DIMENSION}

{'dense_vectors': 7, 'dimension': 32}

In [8]:
graph_out: dict[str, list[tuple[str, str, str]]] = defaultdict(list)
graph_in: dict[str, list[tuple[str, str, str]]] = defaultdict(list)
for source, relation, target, provenance in relationships:
    graph_out[source].append((relation, target, provenance))
    graph_in[target].append((relation, source, provenance))

{"nodes": len(record_ids), "edges": len(relationships)}

{'nodes': 7, 'edges': 7}

In [9]:
record_store: dict[str, dict[str, Any]] = {}
chunk_store: dict[str, dict[str, Any]] = {}
edge_store: dict[tuple[str, str, str], str] = {}

def publish() -> dict[str, int]:
    created = unchanged = 0
    for key, value in [
        *((record["record_id"], record) for record in records),
        *((chunk["chunk_id"], chunk) for chunk in chunks),
        *((f"{source}|{relation}|{target}", provenance) for source, relation, target, provenance in relationships),
    ]:
        store = (
            record_store
            if key in record_ids
            else edge_store
            if "|" in key
            else chunk_store
        )
        store_key = tuple(key.split("|")) if "|" in key else key
        if store_key in store and store[store_key] == value:
            unchanged += 1
        else:
            created += int(store_key not in store)
            store[store_key] = value
    return {"created": created, "unchanged": unchanged}

first_publish = publish()
second_publish = publish()
idempotent_indexing = first_publish["created"] > 0 and second_publish["created"] == 0
{"first": first_publish, "second": second_publish, "restart_safe": idempotent_indexing}

{'first': {'created': 21, 'unchanged': 0},
 'second': {'created': 0, 'unchanged': 21},
 'restart_safe': True}

# 2. Hybrid retrieval and rerank

![Data retrieval architecture](../docs/images/data-retrieval-architecture.png)

*Exact, word-based, meaning-based, and graph search build a small evidence set before answering.*

We ask three independent questions of the index:

1. **Exact:** does the query name a record identifier?
2. **Sparse:** which chunks share important words?
3. **Dense:** which chunks have similar small meaning vectors?

**Reciprocal rank fusion** combines the three ordered lists. A transparent rerank then rewards exact IDs and current records while demoting superseded evidence.

In [10]:
def exact_ranking(question: str) -> list[str]:
    identifiers = set(IDENTIFIER.findall(question.upper()))
    return [
        chunk["chunk_id"]
        for chunk in chunks
        if chunk["record_id"] in identifiers
    ]

def sparse_ranking(question: str) -> list[tuple[str, float]]:
    query_tokens = TOKEN.findall(question.lower())
    total_documents = len(chunks)
    scored = []
    for chunk_id, tokens in tokenized_chunks.items():
        counts = Counter(tokens)
        score = 0.0
        for token in query_tokens:
            if not counts[token]:
                continue
            inverse_frequency = math.log(
                1 + (total_documents - document_frequency[token] + 0.5)
                / (document_frequency[token] + 0.5)
            )
            length_adjustment = counts[token] + 1.2 * (
                0.25 + 0.75 * len(tokens) / average_length
            )
            score += inverse_frequency * counts[token] * 2.2 / length_adjustment
        scored.append((chunk_id, score))
    return sorted(
        ((chunk_id, score) for chunk_id, score in scored if score > 0),
        key=lambda item: (-item[1], item[0]),
    )

In [11]:
def dense_ranking(question: str) -> list[tuple[str, float]]:
    query_vector = dense_vector(question)
    scores = [
        (
            chunk_id,
            sum(left * right for left, right in zip(query_vector, vector, strict=True)),
        )
        for chunk_id, vector in dense_index.items()
    ]
    return sorted(scores, key=lambda item: (-item[1], item[0]))

In [12]:
def reciprocal_rank_fusion(*rankings: list[str], rank_constant: int = 60) -> dict[str, float]:
    fused: dict[str, float] = defaultdict(float)
    for ranking in rankings:
        for rank, chunk_id in enumerate(ranking, start=1):
            fused[chunk_id] += 1 / (rank_constant + rank)
    return dict(sorted(fused.items(), key=lambda item: (-item[1], item[0])))

In [13]:
def rerank(question: str, fused: dict[str, float]) -> list[dict[str, Any]]:
    by_chunk = {chunk["chunk_id"]: chunk for chunk in chunks}
    query_tokens = set(TOKEN.findall(question.lower()))
    ranked = []
    for chunk_id, fusion_score in fused.items():
        chunk = by_chunk[chunk_id]
        score = fusion_score
        reasons = ["rank fusion"]
        if chunk["record_id"] in question.upper():
            score += 2.0
            reasons.append("exact identifier")
        overlap = len(query_tokens & set(tokenized_chunks[chunk_id])) / max(len(query_tokens), 1)
        score += 0.2 * overlap
        if overlap:
            reasons.append("question-word overlap")
        if chunk["status"] in {"current", "closed", "mitigated"}:
            score += 0.15
            reasons.append("current or resolved status")
        if chunk["status"] == "superseded":
            score -= 0.2
            reasons.append("superseded evidence")
        ranked.append(
            {
                "chunk": chunk,
                "fusion_score": fusion_score,
                "rerank_score": score,
                "reasons": reasons,
            }
        )
    return sorted(ranked, key=lambda item: (-item["rerank_score"], item["chunk"]["chunk_id"]))

In [14]:
class EvidenceItem(BaseModel):
    chunk_id: str
    record_id: str
    text: str
    origin: str
    fusion_score: float
    rerank_score: float
    reasons: list[str]

class EvidencePacket(BaseModel):
    question: str
    evidence: list[EvidenceItem] = Field(default_factory=list)
    exact_candidates: int
    sparse_candidates: int
    dense_candidates: int
    supported: bool

def retrieve(question: str, top_k: int = 5) -> EvidencePacket:
    exact = exact_ranking(question)
    sparse = sparse_ranking(question)
    dense = dense_ranking(question)
    fused = reciprocal_rank_fusion(
        exact,
        [chunk_id for chunk_id, _ in sparse],
        [chunk_id for chunk_id, _ in dense],
    )
    ranked = rerank(question, fused)
    supported = bool(exact or sparse)
    return EvidencePacket(
        question=question,
        evidence=[
            EvidenceItem(
                chunk_id=item["chunk"]["chunk_id"],
                record_id=item["chunk"]["record_id"],
                text=item["chunk"]["text"],
                origin=item["chunk"]["origin"],
                fusion_score=item["fusion_score"],
                rerank_score=item["rerank_score"],
                reasons=item["reasons"],
            )
            for item in ranked[:top_k]
        ],
        exact_candidates=len(exact),
        sparse_candidates=len(sparse),
        dense_candidates=len(dense),
        supported=supported,
    )

In [15]:
example_one_question = "What did RFI-087 decide, and which drawing revision contains it?"
example_one_packet = retrieve(example_one_question)
[
    {
        "record_id": item.record_id,
        "fusion": round(item.fusion_score, 4),
        "rerank": round(item.rerank_score, 4),
        "reasons": item.reasons,
        "origin": item.origin,
    }
    for item in example_one_packet.evidence
]

[{'record_id': 'RFI-087',
  'fusion': 0.0487,
  'rerank': 2.2387,
  'reasons': ['rank fusion',
   'exact identifier',
   'question-word overlap',
   'current or resolved status'],
  'origin': 'synthetic_academic_demo'},
 {'record_id': 'DRAW-S-204-R5',
  'fusion': 0.0318,
  'rerank': 0.2018,
  'reasons': ['rank fusion',
   'question-word overlap',
   'current or resolved status'],
  'origin': 'synthetic_academic_demo'},
 {'record_id': 'SPEC-STEEL-001',
  'fusion': 0.0308,
  'rerank': 0.2008,
  'reasons': ['rank fusion',
   'question-word overlap',
   'current or resolved status'],
  'origin': 'synthetic_academic_demo'},
 {'record_id': 'RISK-DELAY-001',
  'fusion': 0.0154,
  'rerank': 0.1654,
  'reasons': ['rank fusion', 'current or resolved status'],
  'origin': 'synthetic_academic_demo'},
 {'record_id': 'ACT-STEEL-009',
  'fusion': 0.0323,
  'rerank': 0.0523,
  'reasons': ['rank fusion', 'question-word overlap'],
  'origin': 'synthetic_academic_demo'}]

# 3. Fast RAG

Fast RAG is the smallest safe path: retrieve once, keep only permitted evidence, and assemble a cited answer. If retrieval has no credible exact or sparse support, it must **abstain**.

In [16]:
class Citation(BaseModel):
    record_id: str
    chunk_id: str
    origin: str

class AnswerResult(BaseModel):
    route: Literal["rag", "graph_rag", "agentic_rag"]
    answer: str
    citations: list[Citation] = Field(default_factory=list)
    grounded: bool
    abstained: bool
    trace: list[dict[str, Any]] = Field(default_factory=list)

def fast_rag(question: str) -> AnswerResult:
    packet = retrieve(question)
    if not packet.supported:
        return AnswerResult(
            route="rag",
            answer="I do not have enough permitted evidence to answer.",
            grounded=True,
            abstained=True,
        )
    exact_ids = set(IDENTIFIER.findall(question.upper()))
    selected = [
        item for item in packet.evidence
        if not exact_ids or item.record_id in exact_ids
    ][:2]
    selected = selected or packet.evidence[:2]
    return AnswerResult(
        route="rag",
        answer="\n\n".join(f"{item.text} [{item.record_id}]" for item in selected),
        citations=[
            Citation(record_id=item.record_id, chunk_id=item.chunk_id, origin=item.origin)
            for item in selected
        ],
        grounded=True,
        abstained=False,
        trace=[
            {"stage": "retrieve", "summary": f"accepted {len(selected)} evidence item(s)"},
            {"stage": "answer", "summary": "every statement has a citation"},
        ],
    )

fast_answer = fast_rag(example_one_question)
fast_answer.model_dump()

{'route': 'rag',
 'answer': 'Connection C17 clarification. RFI-087 approved plate PL-17B for connection C17. The decision was incorporated in S-204 revision 5. [RFI-087]',
 'citations': [{'record_id': 'RFI-087',
   'chunk_id': 'RFI-087-chunk-1',
   'origin': 'synthetic_academic_demo'}],
 'grounded': True,
 'abstained': False,
 'trace': [{'stage': 'retrieve', 'summary': 'accepted 1 evidence item(s)'},
  {'stage': 'answer', 'summary': 'every statement has a citation'}]}

# 4. Graph RAG

Text retrieval finds passages. Graph RAG answers a different question: **what is connected, by which stored relationship, and according to which source?**

In [17]:
def graph_paths(start_id: str, max_depth: int = 2) -> list[dict[str, Any]]:
    if start_id not in record_ids:
        return []
    queue = deque([(start_id, [start_id], [])])
    paths = []
    while queue:
        current, nodes, edges = queue.popleft()
        if len(edges) >= max_depth:
            continue
        for relation, target, provenance in sorted(graph_out[current]):
            if target in nodes:
                continue
            next_nodes = [*nodes, target]
            next_edges = [*edges, {"source": current, "relation": relation, "target": target, "provenance": provenance}]
            paths.append({"nodes": next_nodes, "edges": next_edges})
            queue.append((target, next_nodes, next_edges))
    return paths

def graph_rag(start_id: str) -> AnswerResult:
    paths = graph_paths(start_id)
    if not paths:
        return AnswerResult(
            route="graph_rag",
            answer="I do not have a verified relationship path for that record.",
            grounded=True,
            abstained=True,
        )
    connected_ids = list(dict.fromkeys(node for path in paths for node in path["nodes"]))
    unique_edges = list(
        {
            (edge["source"], edge["relation"], edge["target"], edge["provenance"]): edge
            for path in paths
            for edge in path["edges"]
        }.values()
    )
    return AnswerResult(
        route="graph_rag",
        answer="; ".join(
            f'{edge["source"]} {edge["relation"]} {edge["target"]} ({edge["provenance"]})'
            for edge in unique_edges
        ),
        citations=[
            Citation(
                record_id=record_id,
                chunk_id=f"{record_id}-chunk-1",
                origin=next(record["origin"] for record in records if record["record_id"] == record_id),
            )
            for record_id in connected_ids
        ],
        grounded=True,
        abstained=False,
        trace=[{"stage": "graph", "summary": f"followed {len(paths)} provenance-backed path(s)"}],
    )

In [18]:
graph_answer = graph_rag("RFI-087")
{
    "route": graph_answer.route,
    "answer": graph_answer.answer,
    "citation_ids": [citation.record_id for citation in graph_answer.citations],
    "paths": graph_paths("RFI-087"),
}

{'route': 'graph_rag',
 'answer': 'RFI-087 AFFECTS ACT-STEEL-009 (RFI response); RFI-087 CHANGES_OR_CLARIFIES DRAW-S-204-R5 (RFI response); RFI-087 REFERENCES DRAW-S-204-R3 (RFI response); ACT-STEEL-009 GOVERNED_BY SPEC-STEEL-001 (schedule mapping); DRAW-S-204-R5 REVISES DRAW-S-204-R3 (drawing register)',
 'citation_ids': ['RFI-087',
  'ACT-STEEL-009',
  'DRAW-S-204-R5',
  'DRAW-S-204-R3',
  'SPEC-STEEL-001'],
 'paths': [{'nodes': ['RFI-087', 'ACT-STEEL-009'],
   'edges': [{'source': 'RFI-087',
     'relation': 'AFFECTS',
     'target': 'ACT-STEEL-009',
     'provenance': 'RFI response'}]},
  {'nodes': ['RFI-087', 'DRAW-S-204-R5'],
   'edges': [{'source': 'RFI-087',
     'relation': 'CHANGES_OR_CLARIFIES',
     'target': 'DRAW-S-204-R5',
     'provenance': 'RFI response'}]},
  {'nodes': ['RFI-087', 'DRAW-S-204-R3'],
   'edges': [{'source': 'RFI-087',
     'relation': 'REFERENCES',
     'target': 'DRAW-S-204-R3',
     'provenance': 'RFI response'}]},
  {'nodes': ['RFI-087', 'ACT-STEEL-0

# 5. Typed tools and specialists

![Typed tools architecture](../docs/images/tools-architecture.png)

*Each tool performs one controlled read-only job and returns sources with its result.*

A tool performs one controlled operation. Here, all five are real LangChain tools created with `@tool`, kept in a small registry, and called through LangChain's `.invoke(...)` interface. A specialist receives only the tools needed for one kind of work:

- **Document Specialist:** search and revision comparison.
- **Schedule Specialist:** activity status and dependency impact.
- **Risk Specialist:** rank risks supported by the observations.

The orchestrator coordinates them; specialists do not become new sources of truth.

In [19]:
class ToolInput(BaseModel):
    tool_name: Literal[
        "search_documents",
        "compare_revisions",
        "query_graph",
        "analyze_schedule",
        "rank_risks",
    ]
    arguments: dict[str, Any] = Field(default_factory=dict)

class ToolObservation(BaseModel):
    tool_name: str
    specialist: str
    success: bool
    summary: str
    evidence_ids: list[str] = Field(default_factory=list)
    data: dict[str, Any] = Field(default_factory=dict)

In [20]:
def _toy_tool_logic(request: ToolInput, specialist: str) -> ToolObservation:
    arguments = request.arguments
    if request.tool_name == "search_documents":
        packet = retrieve(str(arguments["question"]))
        evidence_ids = [item.record_id for item in packet.evidence]
        selected_records = [
            record for record in records if record["record_id"] in evidence_ids
        ]
        return ToolObservation(
            tool_name=request.tool_name,
            specialist=specialist,
            success=packet.supported,
            summary=f"Retrieved {len(packet.evidence)} ranked passage(s).",
            evidence_ids=evidence_ids,
            data={
                "supported": packet.supported,
                "record_ids": evidence_ids,
                "schedule_activity_ids": [
                    record["record_id"]
                    for record in selected_records
                    if record["record_type"] == "schedule_activity"
                ],
                "rfi_ids": [
                    record["record_id"]
                    for record in selected_records
                    if record["record_type"] == "rfi"
                ],
                "drawing_numbers": list(
                    dict.fromkeys(
                        str(record["metadata"]["document_number"])
                        for record in selected_records
                        if record["record_type"] == "drawing"
                    )
                ),
            },
        )
    if request.tool_name == "compare_revisions":
        drawing = str(arguments["document_number"])
        matches = sorted(
            (
                record for record in records
                if record["record_type"] == "drawing"
                and record["metadata"].get("document_number") == drawing
            ),
            key=lambda record: record["revision"],
        )
        current = next(
            (record for record in matches if record["status"] == "current"),
            matches[-1] if matches else None,
        )
        return ToolObservation(
            tool_name=request.tool_name,
            specialist=specialist,
            success=bool(matches),
            summary=f"Compared {len(matches)} controlled revision(s) for {drawing}.",
            evidence_ids=[record["record_id"] for record in matches],
            data={
                "document_number": drawing,
                "revisions": [record["revision"] for record in matches],
                "current_revision": current["revision"] if current else None,
                "current_revision_id": current["record_id"] if current else None,
            },
        )
    if request.tool_name == "query_graph":
        start_id = str(arguments["start_id"])
        paths = graph_paths(start_id, max_depth=3)
        return ToolObservation(
            tool_name=request.tool_name,
            specialist=specialist,
            success=bool(paths),
            summary=f"Found {len(paths)} verified project path(s).",
            evidence_ids=list(dict.fromkeys(node for path in paths for node in path["nodes"])),
            data={"start_id": start_id, "paths": paths},
        )
    if request.tool_name == "analyze_schedule":
        activity_id = str(arguments["activity_id"])
        activity = next(
            (record for record in records if record["record_id"] == activity_id),
            None,
        )
        if activity is None or activity["record_type"] != "schedule_activity":
            return ToolObservation(
                tool_name=request.tool_name,
                specialist=specialist,
                success=False,
                summary=f"No permitted schedule activity named {activity_id}.",
            )
        data = {
            "activity_id": activity_id,
            "status": activity["status"],
            "critical": bool(activity["metadata"].get("critical")),
            "duration_days": activity["metadata"].get("duration_days"),
            "blocked_by_ids": list(activity["metadata"].get("blocked_by_ids", [])),
            "drawing_numbers": list(activity["metadata"].get("drawing_numbers", [])),
        }
        return ToolObservation(
            tool_name=request.tool_name,
            specialist=specialist,
            success=True,
            summary=(
                f'{activity_id} status is {data["status"]}; '
                f'critical={data["critical"]}; duration={data["duration_days"]} days.'
            ),
            evidence_ids=[activity_id],
            data=data,
        )
    seed_ids = set(arguments.get("evidence_ids", []))
    connected_risk_ids = {
        source if source.startswith("RISK-") else target
        for source, _relation, target, _provenance in relationships
        if (source in seed_ids or target in seed_ids)
        and (source.startswith("RISK-") or target.startswith("RISK-"))
    }
    risk_records = [
        record for record in records
        if record["record_type"] == "risk"
        and (not seed_ids or record["record_id"] in connected_risk_ids)
    ]
    risk_rows = [
        {
            "record_id": record["record_id"],
            "severity": record["metadata"].get("severity", "unknown"),
            "status": record["status"],
        }
        for record in risk_records
    ]
    return ToolObservation(
        tool_name=request.tool_name,
        specialist=specialist,
        success=bool(risk_records),
        summary=f"Ranked {len(risk_records)} evidence-backed delay risk(s).",
        evidence_ids=[record["record_id"] for record in risk_records],
        data={"risks": risk_rows},
    )

In [21]:
def _toy_tool_payload(observation: ToolObservation) -> dict[str, Any]:
    return observation.model_dump(exclude={"tool_name", "specialist"})

@tool
def search_documents(question: str) -> dict[str, Any]:
    """Retrieve ranked, permitted toy project passages for one question."""
    return _toy_tool_payload(
        _toy_tool_logic(
            ToolInput(tool_name="search_documents", arguments={"question": question}),
            "Document Specialist",
        )
    )

@tool
def compare_revisions(document_number: str) -> dict[str, Any]:
    """Compare controlled revisions for one drawing number."""
    return _toy_tool_payload(
        _toy_tool_logic(
            ToolInput(
                tool_name="compare_revisions",
                arguments={"document_number": document_number},
            ),
            "Document Specialist",
        )
    )

@tool
def query_graph(start_id: str) -> dict[str, Any]:
    """Follow verified toy project relationships from one record."""
    return _toy_tool_payload(
        _toy_tool_logic(
            ToolInput(tool_name="query_graph", arguments={"start_id": start_id}),
            "Document Specialist",
        )
    )

@tool
def analyze_schedule(activity_id: str) -> dict[str, Any]:
    """Read status and blockers for one toy schedule activity."""
    return _toy_tool_payload(
        _toy_tool_logic(
            ToolInput(
                tool_name="analyze_schedule",
                arguments={"activity_id": activity_id},
            ),
            "Schedule Specialist",
        )
    )

@tool
def rank_risks(evidence_ids: list[str]) -> dict[str, Any]:
    """Rank toy delay risks connected to observed evidence IDs."""
    return _toy_tool_payload(
        _toy_tool_logic(
            ToolInput(tool_name="rank_risks", arguments={"evidence_ids": evidence_ids}),
            "Risk Specialist",
        )
    )

TOOL_REGISTRY: dict[str, BaseTool] = {
    item.name: item
    for item in (
        search_documents,
        compare_revisions,
        query_graph,
        analyze_schedule,
        rank_risks,
    )
}

def call_tool(request: ToolInput, specialist: str) -> ToolObservation:
    payload = TOOL_REGISTRY[request.tool_name].invoke(request.arguments)
    return ToolObservation(
        tool_name=request.tool_name,
        specialist=specialist,
        **payload,
    )

SPECIALISTS = {
    "Document Specialist": ["search_documents", "compare_revisions", "query_graph"],
    "Schedule Specialist": ["analyze_schedule", "query_graph"],
    "Risk Specialist": ["rank_risks"],
}
{"registered_tools": sorted(TOOL_REGISTRY), "specialists": SPECIALISTS}

{'registered_tools': ['analyze_schedule',
  'compare_revisions',
  'query_graph',
  'rank_risks',
  'search_documents'],
 'specialists': {'Document Specialist': ['search_documents',
   'compare_revisions',
   'query_graph'],
  'Schedule Specialist': ['analyze_schedule', 'query_graph'],
  'Risk Specialist': ['rank_risks']}}

# 6. Native LangChain ReAct with `create_agent()`

![Agent orchestration architecture](../docs/images/agent-orchestration-architecture.png)

*LangChain owns the model → tool → observation loop. The notebook does not hand-wire that loop.*

The agent below is built with LangChain's real `create_agent()` primitive—the same pattern used by the course notebooks. Its five operations are the `@tool` objects defined above.

For repeatable offline teaching, the chat model is deterministic: it makes its next tool decision from the returned `ToolMessage` observations, without a network call. **The agent runtime itself is genuine LangChain/LangGraph.** Notebook 07 imports the production runtime, where local/live mode can use the configured network model.

The returned `result["messages"]` is the execution record. We render it as a readable **Plan → Act → Observe → Decide** timeline: model decision → tool name and arguments → tool observation and evidence → final answer. This is not private hidden chain-of-thought.

In [22]:
ALLOWED_PREFERENCES = {
    "answer_style": {"concise", "detailed", "plain_language"},
    "citation_detail": {"compact", "expanded"},
}

class PreferenceMemory:
    def __init__(self) -> None:
        self.values: dict[tuple[str, str], str] = {}

    def add(self, user_id: str, preference_type: str, value: str) -> None:
        normalized = value.strip().lower()
        if normalized not in ALLOWED_PREFERENCES.get(preference_type, set()):
            raise ValueError("Memory accepts only an allowlisted presentation preference.")
        if IDENTIFIER.search(value.upper()):
            raise ValueError("Project facts must never be stored as preferences.")
        self.values[(user_id, preference_type)] = normalized

    def get(self, user_id: str) -> dict[str, str]:
        return {
            preference_type: value
            for (stored_user, preference_type), value in self.values.items()
            if stored_user == user_id
        }

memory = PreferenceMemory()
memory.add("course-reviewer", "answer_style", "concise")
memory.add("course-reviewer", "citation_detail", "expanded")
memory.get("course-reviewer")

{'answer_style': 'concise', 'citation_detail': 'expanded'}

In [23]:
TOOL_SPECIALIST = {
    "search_documents": "Document Specialist",
    "compare_revisions": "Document Specialist",
    "query_graph": "Document Specialist",
    "analyze_schedule": "Schedule Specialist",
    "rank_risks": "Risk Specialist",
}


def decode_tool_payload(content: Any) -> dict[str, Any]:
    """Turn LangChain ToolMessage content into the tool's structured result."""
    if isinstance(content, dict):
        return content
    if isinstance(content, str):
        try:
            decoded = json.loads(content)
        except (TypeError, ValueError):
            return {"success": False, "summary": content, "evidence_ids": [], "data": {}}
        return decoded if isinstance(decoded, dict) else {"data": {"value": decoded}}
    return {"success": False, "summary": str(content), "evidence_ids": [], "data": {}}


class EducationalToolCallingModel(BaseChatModel):
    """Deterministic chat model; `create_agent` still owns the real agent loop."""

    bound_tool_names: frozenset[str] = Field(default_factory=frozenset, exclude=True)

    @property
    def _llm_type(self) -> str:
        return "educational-deterministic-tool-calling"

    def bind_tools(
        self,
        tools: Sequence[BaseTool | dict[str, Any] | type | Any],
        *,
        tool_choice: str | None = None,
        **kwargs: Any,
    ) -> "EducationalToolCallingModel":
        names = frozenset(tool.name for tool in tools if isinstance(tool, BaseTool))
        return self.model_copy(update={"bound_tool_names": names})

    @staticmethod
    def _tool_call(name: str, arguments: dict[str, Any], number: int) -> ChatResult:
        return ChatResult(
            generations=[
                ChatGeneration(
                    message=AIMessage(
                        content="",
                        tool_calls=[
                            {
                                "name": name,
                                "args": arguments,
                                "id": f"toy-call-{number}-{name}",
                                "type": "tool_call",
                            }
                        ],
                    )
                )
            ]
        )

    @staticmethod
    def _observations(messages: list[BaseMessage]) -> list[dict[str, Any]]:
        return [
            {"tool_name": message.name or "unknown", **decode_tool_payload(message.content)}
            for message in messages
            if isinstance(message, ToolMessage)
        ]

    @staticmethod
    def _final_answer(observations: list[dict[str, Any]]) -> str:
        evidence_ids = list(
            dict.fromkeys(
                evidence_id
                for observation in observations
                for evidence_id in observation.get("evidence_ids", [])
            )
        )
        if not evidence_ids:
            return "I do not have enough permitted evidence to answer."

        by_tool = {observation["tool_name"]: observation for observation in observations}
        statements: list[str] = []
        schedule = by_tool.get("analyze_schedule", {}).get("data", {})
        if schedule:
            importance = "critical" if schedule.get("critical") else "not marked critical"
            blockers = ", ".join(schedule.get("blocked_by_ids", [])) or "no recorded blocker"
            statements.append(
                f"{schedule.get('activity_id')} is {schedule.get('status')}, has a planned "
                f"duration of {schedule.get('duration_days')} days, is {importance}, and was "
                f"blocked by {blockers}."
            )
        revision = by_tool.get("compare_revisions", {}).get("data", {})
        if revision:
            statements.append(
                f"The controlled drawing register identifies {revision.get('document_number')} "
                f"revision {revision.get('current_revision')} as current."
            )
        graph = by_tool.get("query_graph", {}).get("data", {})
        if graph:
            statements.append(
                f"The project graph verifies {len(graph.get('paths', []))} connected path(s) "
                f"from {graph.get('start_id')}."
            )
        risks = by_tool.get("rank_risks", {}).get("data", {}).get("risks", [])
        if risks:
            statements.append(
                f"The connected risk {risks[0]['record_id']} is {risks[0]['severity']} "
                f"and {risks[0]['status']}."
            )
        citations = " ".join(f"[{record_id}]" for record_id in evidence_ids)
        return " ".join(statements) + f"\n\nSources: {citations}"

    def _generate(
        self,
        messages: list[BaseMessage],
        stop: list[str] | None = None,
        run_manager: Any = None,
        **kwargs: Any,
    ) -> ChatResult:
        question = next(
            (str(message.content) for message in messages if isinstance(message, HumanMessage)),
            "",
        )
        observations = self._observations(messages)
        used = {observation["tool_name"] for observation in observations}
        by_tool = {observation["tool_name"]: observation for observation in observations}
        call_number = len(observations) + 1

        if "search_documents" not in used:
            return self._tool_call("search_documents", {"question": question}, call_number)

        search_data = by_tool["search_documents"].get("data", {})
        schedule_ids = list(search_data.get("schedule_activity_ids", []))
        rfi_ids = list(search_data.get("rfi_ids", []))
        if schedule_ids and "analyze_schedule" not in used:
            return self._tool_call(
                "analyze_schedule", {"activity_id": schedule_ids[0]}, call_number
            )

        schedule_data = by_tool.get("analyze_schedule", {}).get("data", {})
        drawing_numbers = list(schedule_data.get("drawing_numbers", []))
        if drawing_numbers and "compare_revisions" not in used:
            return self._tool_call(
                "compare_revisions", {"document_number": drawing_numbers[0]}, call_number
            )

        graph_start_ids = list(schedule_data.get("blocked_by_ids", [])) or rfi_ids
        if graph_start_ids and "query_graph" not in used:
            return self._tool_call(
                "query_graph", {"start_id": graph_start_ids[0]}, call_number
            )

        if "query_graph" in used and "rank_risks" not in used:
            evidence_ids = list(
                dict.fromkeys(
                    evidence_id
                    for observation in observations
                    for evidence_id in observation.get("evidence_ids", [])
                )
            )
            return self._tool_call(
                "rank_risks", {"evidence_ids": evidence_ids}, call_number
            )

        return ChatResult(
            generations=[
                ChatGeneration(message=AIMessage(content=self._final_answer(observations)))
            ]
        )


SYSTEM_PROMPT = """You are an evidence-first civil engineering project assistant.
Use only the five supplied read-only tools. Start with document search, inspect each
returned observation, and call only the next tool supported by that evidence. Stop
when the delay, drawing change, relationship path, and connected risk are evidenced.
Never invent a source identifier."""

TOY_AGENT = create_agent(
    model=EducationalToolCallingModel(),
    tools=list(TOOL_REGISTRY.values()),
    system_prompt=SYSTEM_PROMPT,
    middleware=[ToolCallLimitMiddleware(run_limit=6, exit_behavior="continue")],
)

example_two_question = (
    "Why was ACT-STEEL-009 blocked, what changed in S-204, "
    "what risk followed, and what evidence closes the issue?"
)
agent_result = TOY_AGENT.invoke(
    {"messages": [{"role": "user", "content": example_two_question}]},
    config={"recursion_limit": 32, "run_name": "educational-native-create-agent"},
)

In [24]:
def agent_trace_records(result: dict[str, Any]) -> list[dict[str, Any]]:
    """Preserve every returned message as a display-safe execution timeline."""
    rows: list[dict[str, Any]] = []
    for message_index, message in enumerate(result["messages"]):
        if isinstance(message, HumanMessage):
            rows.append(
                {
                    "kind": "question",
                    "title": "Question",
                    "content": str(message.content),
                    "message_index": message_index,
                }
            )
            continue
        if isinstance(message, AIMessage):
            calls = list(message.tool_calls or [])
            if calls:
                rows.append(
                    {
                        "kind": "model_response",
                        "title": "Model decision",
                        "content": f"Selected {len(calls)} next tool call(s) after the available observations.",
                        "message_index": message_index,
                    }
                )
                for call in calls:
                    rows.append(
                        {
                            "kind": "tool_call",
                            "title": f"Tool call · {call['name']}",
                            "content": TOOL_SPECIALIST[call["name"]],
                            "tool_name": call["name"],
                            "arguments": call.get("args", {}),
                            "message_index": message_index,
                        }
                    )
            elif str(message.content).strip():
                rows.append(
                    {
                        "kind": "final_answer",
                        "title": "Final grounded answer",
                        "content": str(message.content),
                        "message_index": message_index,
                    }
                )
            continue
        if isinstance(message, ToolMessage):
            payload = decode_tool_payload(message.content)
            rows.append(
                {
                    "kind": "tool_observation",
                    "title": f"Tool observation · {message.name}",
                    "content": payload.get("summary", "Structured observation returned."),
                    "tool_name": message.name,
                    "evidence_ids": payload.get("evidence_ids", []),
                    "observation": payload,
                    "message_index": message_index,
                }
            )
    return rows


def render_agent_trace(result: dict[str, Any]) -> list[dict[str, Any]]:
    """Render the native LangChain message history as compact notebook cards."""
    rows = agent_trace_records(result)
    palette = {
        "question": ("#0f172a", "#e0f2fe", "❓"),
        "model_response": ("#4c1d95", "#ede9fe", "🧠"),
        "tool_call": ("#1d4ed8", "#dbeafe", "🔧"),
        "tool_observation": ("#166534", "#dcfce7", "📄"),
        "final_answer": ("#92400e", "#fef3c7", "✅"),
    }
    cards: list[str] = []
    for step, row in enumerate(rows, start=1):
        color, background, icon = palette[row["kind"]]
        details: list[str] = []
        if "arguments" in row:
            details.append(
                "<div class='trace-label'>Arguments</div>"
                f"<pre>{escape(json.dumps(row['arguments'], indent=2, sort_keys=True))}</pre>"
            )
        evidence_ids = row.get("evidence_ids", [])
        if evidence_ids:
            chips = "".join(
                f"<span class='trace-chip'>{escape(str(source_id))}</span>"
                for source_id in evidence_ids
            )
            details.append(f"<div class='trace-label'>Evidence</div><div>{chips}</div>")
        if "observation" in row:
            details.append(
                "<details><summary>Full structured observation</summary>"
                f"<pre>{escape(json.dumps(row['observation'], indent=2, sort_keys=True))}</pre>"
                "</details>"
            )
        content = escape(str(row["content"])).replace("\n", "<br>")
        cards.append(
            f"<section class='trace-card' style='border-left-color:{color};background:{background}'>"
            f"<div class='trace-step'>STEP {step:02d} · MESSAGE {row['message_index']:02d}</div>"
            f"<h4 style='color:{color}'>{icon} {escape(row['title'])}</h4>"
            f"<div class='trace-content'>{content}</div>{''.join(details)}</section>"
        )
    html = """
    <style>
      .native-agent-trace {font-family:Inter,ui-sans-serif,system-ui,sans-serif;max-width:1050px}
      .trace-heading {font-size:22px;font-weight:800;color:#0f172a;margin:4px 0 14px}
      .trace-card {border-left:6px solid;border-radius:14px;margin:12px 0;padding:16px 18px;
                   box-shadow:0 3px 12px rgba(15,23,42,.08)}
      .trace-card h4 {margin:2px 0 9px;font-size:17px}
      .trace-step {font-size:10px;font-weight:800;letter-spacing:.12em;color:#64748b}
      .trace-content {color:#1e293b;line-height:1.55}
      .trace-label {font-size:11px;font-weight:800;letter-spacing:.08em;color:#475569;
                    text-transform:uppercase;margin:12px 0 5px}
      .trace-card pre {white-space:pre-wrap;background:#0f172a;color:#e2e8f0;padding:11px;
                       border-radius:9px;font-size:12px;overflow:auto}
      .trace-chip {display:inline-block;background:#fff;border:1px solid #86efac;border-radius:999px;
                   padding:3px 9px;margin:2px 5px 2px 0;font-size:12px;font-weight:700;color:#166534}
      .trace-card details {margin-top:10px;color:#334155}
      .trace-card summary {cursor:pointer;font-weight:700}
    </style>
    <div class="native-agent-trace">
      <div class="trace-heading">Native LangChain agent execution trace</div>
    """ + "".join(cards) + "</div>"
    display(HTML(html))
    return rows


def answer_from_agent_result(result: dict[str, Any], user_id: str) -> AnswerResult:
    rows = agent_trace_records(result)
    observations = [
        row["observation"] for row in rows if row["kind"] == "tool_observation"
    ]
    evidence_ids = list(
        dict.fromkeys(
            evidence_id
            for observation in observations
            for evidence_id in observation.get("evidence_ids", [])
        )
    )
    final_row = next(
        (row for row in reversed(rows) if row["kind"] == "final_answer"), None
    )
    abstained = not evidence_ids or final_row is None
    trace = [
        {
            "stage": "memory",
            "summary": f"Loaded {len(memory.get(user_id))} allowlisted preference(s).",
        }
    ]
    for row in rows:
        if row["kind"] == "model_response":
            trace.append({"stage": "plan", "summary": row["content"]})
        elif row["kind"] == "tool_call":
            trace.append(
                {
                    "stage": "act",
                    "summary": f"{row['content']} called {row['tool_name']}.",
                    "tool_name": row["tool_name"],
                    "arguments": row["arguments"],
                }
            )
        elif row["kind"] == "tool_observation":
            trace.append(
                {
                    "stage": "observe",
                    "summary": row["content"],
                    "tool_name": row["tool_name"],
                    "data": row["observation"].get("data", {}),
                }
            )
        elif row["kind"] == "final_answer":
            trace.append({"stage": "decide", "summary": "Evidence is sufficient; stop."})
    return AnswerResult(
        route="agentic_rag",
        answer=(
            "I do not have enough permitted evidence to answer."
            if abstained
            else str(final_row["content"])
        ),
        citations=[
            Citation(
                record_id=record_id,
                chunk_id=f"{record_id}-chunk-1",
                origin=next(
                    record["origin"] for record in records if record["record_id"] == record_id
                ),
            )
            for record_id in evidence_ids
            if record_id in record_ids
        ],
        grounded=True,
        abstained=abstained,
        trace=trace,
    )


agent_trace_rows = render_agent_trace(agent_result)
agent_answer = answer_from_agent_result(agent_result, "course-reviewer")
{
    "route": agent_answer.route,
    "answer": agent_answer.answer,
    "citation_ids": [citation.record_id for citation in agent_answer.citations],
    "preferences": memory.get("course-reviewer"),
    "native_message_count": len(agent_result["messages"]),
}

{'route': 'agentic_rag',
 'answer': 'ACT-STEEL-009 is in_progress, has a planned duration of 7 days, is critical, and was blocked by RFI-087. The controlled drawing register identifies S-204 revision 5 as current. The project graph verifies 5 connected path(s) from RFI-087. The connected risk RISK-DELAY-001 is high and mitigated.\n\nSources: [ACT-STEEL-009] [RFI-087] [RISK-DELAY-001] [SPEC-STEEL-001] [DRAW-S-204-R5] [DRAW-S-204-R3]',
 'citation_ids': ['ACT-STEEL-009',
  'RFI-087',
  'RISK-DELAY-001',
  'SPEC-STEEL-001',
  'DRAW-S-204-R5',
  'DRAW-S-204-R3'],
 'preferences': {'answer_style': 'concise', 'citation_detail': 'expanded'},
 'native_message_count': 12}

# 7. Safe observability

The visual timeline above is built directly from LangChain's returned `result["messages"]`. It shows each model turn, tool name, validated arguments, structured observation, evidence identifiers, and final answer.

It deliberately does **not** claim to expose private hidden chain-of-thought. In the production application, the same kind of structured events are also associated with a run-specific trace reference.

In [25]:
def recall_at_k(retrieved: list[str], expected: set[str], k: int) -> float:
    return len(set(retrieved[:k]) & expected) / max(len(expected), 1)

def citation_coverage(answer: AnswerResult) -> float:
    if answer.abstained:
        return float(not answer.citations)
    return float(answer.grounded and bool(answer.citations))

def tool_selection_precision(trace: list[dict[str, Any]], expected_tools: set[str]) -> float:
    actual = {
        summary.rsplit(" ", 1)[-1].rstrip(".")
        for summary in (event["summary"] for event in trace if event["stage"] == "act")
    }
    return len(actual & expected_tools) / max(len(actual), 1)

In [26]:
gold_examples = [
    {
        "name": "exact RFI lookup",
        "question": example_one_question,
        "answer": fast_answer,
        "expected_route": "rag",
        "expected_evidence": {"RFI-087"},
    },
    {
        "name": "verified downstream path",
        "question": "What is downstream of RFI-087 if its decision is not implemented?",
        "answer": graph_answer,
        "expected_route": "graph_rag",
        "expected_evidence": {"RFI-087", "ACT-STEEL-009"},
    },
    {
        "name": "bounded delay investigation",
        "question": example_two_question,
        "answer": agent_answer,
        "expected_route": "agentic_rag",
        "expected_evidence": {"ACT-STEEL-009", "RFI-087", "DRAW-S-204-R5"},
    },
]

evaluation = []
for example in gold_examples:
    answer = example["answer"]
    citation_ids = [citation.record_id for citation in answer.citations]
    recall = recall_at_k(citation_ids, example["expected_evidence"], 6)
    coverage = citation_coverage(answer)
    within_limit = len(
        [event for event in answer.trace if event["stage"] == "act"]
    ) <= 6
    passed = (
        answer.route == example["expected_route"]
        and answer.grounded
        and bool(citation_ids)
        and recall == 1.0
        and coverage == 1.0
        and within_limit
    )
    evaluation.append(
        {
            "example": example["name"],
            "question": example["question"],
            "route": answer.route,
            "route_correct": answer.route == example["expected_route"],
            "grounded": answer.grounded,
            "citation_ids": citation_ids,
            "recall_at_6": recall,
            "citation_coverage": coverage,
            "within_step_limit": within_limit,
            "evaluation_passed": passed,
        }
    )
route_eval_matrix = {
    example["expected_route"]: result
    for example, result in zip(gold_examples, evaluation, strict=True)
}
assert set(route_eval_matrix) == {"rag", "graph_rag", "agentic_rag"}
assert all(item["evaluation_passed"] for item in route_eval_matrix.values())
print("ROUTE_EVAL_MATRIX " + json.dumps(route_eval_matrix, sort_keys=True))

ROUTE_EVAL_MATRIX {"agentic_rag": {"citation_coverage": 1.0, "citation_ids": ["ACT-STEEL-009", "RFI-087", "RISK-DELAY-001", "SPEC-STEEL-001", "DRAW-S-204-R5", "DRAW-S-204-R3"], "evaluation_passed": true, "example": "bounded delay investigation", "grounded": true, "question": "Why was ACT-STEEL-009 blocked, what changed in S-204, what risk followed, and what evidence closes the issue?", "recall_at_6": 1.0, "route": "agentic_rag", "route_correct": true, "within_step_limit": true}, "graph_rag": {"citation_coverage": 1.0, "citation_ids": ["RFI-087", "ACT-STEEL-009", "DRAW-S-204-R5", "DRAW-S-204-R3", "SPEC-STEEL-001"], "evaluation_passed": true, "example": "verified downstream path", "grounded": true, "question": "What is downstream of RFI-087 if its decision is not implemented?", "recall_at_6": 1.0, "route": "graph_rag", "route_correct": true, "within_step_limit": true}, "rag": {"citation_coverage": 1.0, "citation_ids": ["RFI-087"], "evaluation_passed": true, "example": "exact RFI lookup",

In [27]:
unsupported = fast_rag("Explain Martian aquifer propulsion ZYXXQ-919.")
specialist_names = {
    event["summary"].split(" called ", 1)[0]
    for event in agent_answer.trace
    if event["stage"] == "act"
}
native_tool_calls = {
    row["tool_name"]: row["arguments"]
    for row in agent_trace_rows
    if row["kind"] == "tool_call"
}
native_trace_kinds = {row["kind"] for row in agent_trace_rows}
observation_driven_arguments = (
    native_tool_calls["analyze_schedule"] == {"activity_id": "ACT-STEEL-009"}
    and native_tool_calls["compare_revisions"] == {"document_number": "S-204"}
    and native_tool_calls["query_graph"] == {"start_id": "RFI-087"}
    and "ACT-STEEL-009" in native_tool_calls["rank_risks"]["evidence_ids"]
    and "RFI-087" in native_tool_calls["rank_risks"]["evidence_ids"]
)
toy_verification = {
    "abstention": unsupported.abstained and not unsupported.citations,
    "agentic_rag": (
        agent_answer.route == "agentic_rag"
        and specialist_names
        == {"Document Specialist", "Schedule Specialist", "Risk Specialist"}
        and len([event for event in agent_answer.trace if event["stage"] == "act"]) <= 6
    ),
    "fast_rag": fast_answer.route == "rag" and not fast_answer.abstained,
    "graph_rag": (
        graph_answer.route == "graph_rag"
        and any(citation.record_id == "ACT-STEEL-009" for citation in graph_answer.citations)
    ),
    "idempotent_indexing": idempotent_indexing,
    "memory": memory.get("course-reviewer")["answer_style"] == "concise",
    "observation_branches": (
        observation_driven_arguments
        and native_trace_kinds
        >= {"question", "model_response", "tool_call", "tool_observation", "final_answer"}
    ),
    "three_route_examples": (
        len(gold_examples) == 3
        and all(item["evaluation_passed"] for item in evaluation)
    ),
}
toy_verification["all_passed"] = all(toy_verification.values())
assert toy_verification["all_passed"]
print(json.dumps(toy_verification, sort_keys=True))

{"abstention": true, "agentic_rag": true, "all_passed": true, "fast_rag": true, "graph_rag": true, "idempotent_indexing": true, "memory": true, "observation_branches": true, "three_route_examples": true}


# 8. How this maps to production

| Educational toy | Production counterpart |
|---|---|
| Inline records and chunks | Validated public and synthetic corpus |
| Python dictionaries | Purpose-specific record, search, and graph stores |
| Token scores and hash vectors | Exact/full-text and embedding retrieval |
| Transparent RRF and rerank | Production fusion and reranking service |
| Breadth-first relationship walk | Permission-aware graph query service |
| Five real LangChain `@tool` objects | Central read-only production tool registry |
| Deterministic chat model + native `create_agent()` | Configured model + native `create_agent()` specialists |
| In-memory allowlisted preferences | Approved preference-memory boundary |
| Beautiful rendering of `result["messages"]` | Structured application and Langfuse traces |
| Three representative route questions | Versioned evaluation suite |

The toy is intentionally small enough to read. Its agent loop is genuinely LangChain/LangGraph, while its offline model and miniature data are explicitly educational rather than production quality claims.

> **EDUCATIONAL TOY — NOT PRODUCTION**

You have now seen the complete pattern: prepare correlated evidence, build several indexes, choose the smallest safe route, use bounded tools for complex work, remember preferences only, cite every supported answer, and abstain when evidence is missing.